# Week 1 · k-Nearest Neighbours: Bird or Drone?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%201/knn_bird_or_drone.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

This notebook repeats the worked example from the Week 1 slide *k-Nearest Neighbours by Hand* and then scales it up to a larger dataset with a train / test split.

**Tasks**
1. Check the hand calculation from the lecture (Part 1).
2. Repeat it with scikit-learn (Part 2).
3. Change `k` and watch the training and test scores (Part 3).

## Part 1 · The five points from the lecture

Two scaled features per object: speed and altitude. Labels: 0 = bird, 1 = drone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[1, 1], [2, 1], [1, 2], [4, 3], [5, 4]], dtype=float)  # A, B, E, C, D
y = np.array([0, 0, 0, 1, 1])
names = ["A", "B", "E", "C", "D"]
classes = ["bird", "drone"]
q = np.array([3.2, 2.2])  # the query object

In [ ]:
d = np.linalg.norm(X - q, axis=1)  # Euclidean distance to every training point
order = np.argsort(d)
for i in order:
    print(f"{names[i]}  distance {d[i]:.3f}  {classes[y[i]]}")

for k in (1, 3):
    votes = np.bincount(y[order[:k]], minlength=2)
    print(f"k = {k}: votes bird {votes[0]}, drone {votes[1]} -> {classes[votes.argmax()]}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
for c, m in ((0, "o"), (1, "^")):
    ax.scatter(*X[y == c].T, marker=m, s=80, label=classes[c])
for (px, py), n in zip(X, names):
    ax.annotate(n, (px + 0.08, py + 0.08))
ax.scatter(*q, marker="*", s=200, c="k", label="query q")
ax.set_xlabel("speed (scaled)"); ax.set_ylabel("altitude (scaled)")
ax.legend(); ax.set_aspect("equal"); plt.show()

**Question.** Why does the answer change between k = 1 and k = 3?

## Part 2 · The same with scikit-learn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in (1, 3, 5):
    model = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    print(f"k = {k}: prediction for q = {classes[model.predict([q])[0]]}")

**Question.** With only five training points, what does k = 5 always predict, and why?

## Part 3 · A larger dataset: choose k with a test set

We simulate 400 radar tracks. Drones fly a little faster and higher than birds, but the two classes overlap, so no classifier is perfect.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

rng = np.random.default_rng(4011)
n = 200
birds = np.column_stack([rng.normal(12, 4, n), rng.normal(40, 20, n)])   # speed m/s, altitude m
drones = np.column_stack([rng.normal(17, 5, n), rng.normal(70, 25, n)])
X_big = np.vstack([birds, drones])
y_big = np.r_[np.zeros(n, int), np.ones(n, int)]

X_tr, X_te, y_tr, y_te = train_test_split(X_big, y_big, test_size=0.3, random_state=0, stratify=y_big)
scaler = StandardScaler().fit(X_tr)          # fit the scaling on training data only
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

print("  k   train acc   test acc   precision   recall   (positive class = drone)")
for k in (1, 3, 5, 11, 25, 51, 151):
    m = KNeighborsClassifier(n_neighbors=k).fit(X_tr_s, y_tr)
    p_tr, p_te = m.predict(X_tr_s), m.predict(X_te_s)
    print(f"{k:4d}   {accuracy_score(y_tr, p_tr):8.3f}   {accuracy_score(y_te, p_te):8.3f}   "
          f"{precision_score(y_te, p_te):9.3f}   {recall_score(y_te, p_te):6.3f}")

**Questions**
1. Why is the training accuracy exactly 1.000 for k = 1? Is that a good sign?
2. Which k would you choose, and why must you choose it *without* looking at the test set in a real project? (Hint: the Week 1 slide *Train, Validate, Test*.)
3. For a drone-detection radar at an airport, which matters more: precision or recall?